# 02. Dynamic Shapes, Tensor Contiguity, v6e MXU (`256 × 256`) Alignment & `bfloat16`

> **Reference Docs**: [Dynamic Shapes](https://torchtpu.readthedocs.io/en/0.2.0/dynamic_shapes/) · [TPU Memory Model](https://torchtpu.readthedocs.io/en/0.2.0/torchtpu_memory_model/) · [Numerical Precision](https://torchtpu.readthedocs.io/en/0.2.0/numerical_precision/)

이 노트북에서는 **TPU v6e (Trillium)** 환경에서 성능을 좌우하는 4가지 하드웨어 정렬 기법을 실습합니다:
1. **Dynamic Shapes 재컴파일 방지**: `dynamic=False` vs `dynamic=None`(128-버킷 **Bounded Dynamism**)
2. **Contiguity & View 비용**: Non-contiguous view 입력 시 발생하는 PjRt 임시 복사와 `.contiguous()` 유지
3. **v6e MXU(`256 × 256`) & 레지스터(`(8, 128)`) 경계 정렬**: 비정렬(`3997`) vs `128` 정렬(`3968`) vs **`256` 정렬(`4096`)** 유효 TFLOPS 비교
4. **`bfloat16` 및 MXU 정밀도 관리**: `bf16` 네이티브 연산 vs `fp32` 1-pass(`DEFAULT`) / 3-pass(`HIGH`) / 6-pass(`HIGHEST`)

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("torch_tpu") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "setuptools<81"])
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "torch_tpu", "xprof", "tensorboard", "tensorboard-plugin-profile", "matplotlib",
    ])
    print("설치 완료! Colab 상단 [런타임 -> 세션 다시 시작] 후 다음 셀부터 실행하세요.")
else:
    print("TorchTPU 환경이 이미 준비되어 있습니다.")

In [ ]:
import os
import time
import torch
from torch._functorch._aot_autograd import autograd_cache

os.environ.setdefault("TPU_VISIBLE_DEVICES", "0")
device = torch.device("tpu")
print(f"Initialized: {torch.tpu.get_device_properties(device).name}")

## 1. Dynamic Shape 재컴파일 방지 (`Bounded Dynamism`)

Autoregressive 디코딩처럼 매 스텝 시퀀스 길이(`dim=2`)가 1씩 늘어나는 워크로드에서:
- `dynamic=False`: 매 스텝 Dynamo shape guard가 깨지며 XLA 전체 재컴파일이 반복됩니다.
- `dynamic=None` (기본 **Bounded Dynamism**): TorchTPU가 `SymInt` 차원을 TPU 벡터 레인 크기인 **`128` 배수 버킷(`[17, 128]`, `[129, 256]`, ...)** 상한선으로 패딩하여 초기 워밍업 이후 재컴파일 없이 실행 파일을 재사용합니다.

In [ ]:
def decode_kv_step(q: torch.Tensor, kv_cache: torch.Tensor, new_kv: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    updated_kv = torch.cat([kv_cache, new_kv], dim=2)
    scores = torch.matmul(q, updated_kv.transpose(-2, -1))
    attn_out = torch.matmul(torch.softmax(scores.float(), dim=-1).to(q.dtype), updated_kv)
    return attn_out, updated_kv

def run_growing_loop(compiled_fn, steps: int = 10) -> list[float]:
    q = torch.randn(1, 8, 1, 128, dtype=torch.bfloat16, device=device)
    kv = torch.randn(1, 8, 16, 128, dtype=torch.bfloat16, device=device)
    new_kv = torch.randn(1, 8, 1, 128, dtype=torch.bfloat16, device=device)
    torch.accelerator.synchronize()

    latencies_ms = []
    for _ in range(steps):
        t0 = time.perf_counter()
        _, kv = compiled_fn(q, kv, new_kv)
        torch.accelerator.synchronize()
        latencies_ms.append((time.perf_counter() - t0) * 1000)
    return latencies_ms

torch._dynamo.reset()
autograd_cache.AOTAutogradCache.clear()
torch._dynamo.config.recompile_limit = 64
static_times = run_growing_loop(torch.compile(decode_kv_step, backend="tpu", dynamic=False))

torch._dynamo.reset()
autograd_cache.AOTAutogradCache.clear()
dynamic_times = run_growing_loop(torch.compile(decode_kv_step, backend="tpu", dynamic=None))

for step_idx, (t_stat, t_dyn) in enumerate(zip(static_times, dynamic_times), 1):
    print(f"Step {step_idx:2d} | dynamic=False: {t_stat:7.2f} ms | dynamic=None (128-bucket): {t_dyn:7.2f} ms")

## 2. 텐서 Contiguity 유지 & View Relayout 오버헤드 제거

TPU 메모리는 1D 바이트 배열이 아니라 **2D 레지스터 정렬 타일**로 관리됩니다.
`torch.compile`로 컴파일된 PjRt 실행 파일은 **contiguous, zero-offset** 버퍼를 요구하므로, 바깥에서 `.transpose()` 같은 non-contiguous view를 그대로 넘기면 매 호출마다 Eager 모드 임시 복사(Relayout)가 발생합니다.

In [ ]:
@torch.compile(backend="tpu")
def compiled_mlp(x_in: torch.Tensor, w: torch.Tensor) -> torch.Tensor:
    return torch.nn.functional.gelu(x_in @ w)

base = torch.randn(4096, 4096, device=device, dtype=torch.bfloat16)
non_contig_view = base.t()                    # is_contiguous() == False
contig_tensor = non_contig_view.contiguous()  # 사전에 1회 contiguous 정렬
weight = torch.randn(4096, 4096, device=device, dtype=torch.bfloat16)

def time_call(inp: torch.Tensor, iters: int = 30) -> float:
    for _ in range(5):
        _ = compiled_mlp(inp, weight)
    torch.accelerator.synchronize()
    t0 = time.perf_counter()
    for _ in range(iters):
        _ = compiled_mlp(inp, weight)
    torch.accelerator.synchronize()
    return (time.perf_counter() - t0) / iters * 1e3

ms_view = time_call(non_contig_view)
ms_contig = time_call(contig_tensor)
print(f"Non-contiguous view input : {ms_view:.3f} ms (매 호출 임시 복사 발생)")
print(f"Contiguous tensor input   : {ms_contig:.3f} ms")

## 3. TPU v6e MXU(`256 × 256`) & VPU(`(8, 128)`) 타일 경계 정렬

TPU v6e(Trillium)는 MXU 크기가 기존 v5e($128 \times 128$)에서 **$256 \times 256$**으로 2배 확장되었습니다.
1. **비정렬(`3997×3999×4001`)**: 레인/타일 패딩 낭비 발생
2. **`128` 홀수 배수 정렬(`3968 = 128 × 31`)**: VPU 레인(`128`)에는 맞지만 $256 \times 256$ MXU 마지막 타일이 절반만 채워짐
3. **`256` 배수 정렬(`4096 = 256 × 16`)**: v6e $256 \times 256$ MXU 타일을 100% 밀도 있게 활용

In [ ]:
@torch.compile(backend="tpu")
def matmul_op(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    return torch.matmul(a, b)

shapes = [
    ("1. Unaligned   (3997x3999x4001)", 3997, 3999, 4001),
    ("2. 128-Aligned (3968x3968x3968)", 3968, 3968, 3968),
    ("3. 256-Aligned (4096x4096x4096)", 4096, 4096, 4096),
]

for label, M, K, N in shapes:
    a = torch.randn(M, K, device=device, dtype=torch.bfloat16)
    b = torch.randn(K, N, device=device, dtype=torch.bfloat16)
    for _ in range(5):
        _ = matmul_op(a, b)
    torch.accelerator.synchronize()

    iters = 30
    t0 = time.perf_counter()
    for _ in range(iters):
        _ = matmul_op(a, b)
    torch.accelerator.synchronize()
    dt = (time.perf_counter() - t0) / iters
    tflops = (2 * M * K * N / dt) / 1e12
    print(f"{label} -> {dt*1e3:6.3f} ms | Effective {tflops:6.1f} TFLOPS")

## 4. `bfloat16` vs `float32` MXU Multi-Pass 정밀도 제어

TPU v6e MXU는 하드웨어적으로 `bf16` 입력 곱셈 + `fp32` 누산기 구조입니다.
`fp32` 텐서를 행렬곱할 때 `torch.tpu.precision` 설정에 따른 속도(TFLOPS)와 수치 오차를 측정합니다:
- `Precision.DEFAULT` (`"medium"`, 1-pass): 상위 `bf16` 비트만 곱함 (권장)
- `Precision.HIGH` (`"high"`, 3-pass): `bf16` 상위/하위 분해 후 3회 MXU 패스 (~3배 비용)
- `Precision.HIGHEST` (`"highest"`, 6-pass): 6회 MXU 패스로 IEEE-754 `fp32` 근사 (~6배 비용)

In [ ]:
M = K = N = 4096  # 256 * 16 (TPU v6e MXU aligned)
a_cpu = torch.randn(M, K, dtype=torch.float32)
b_cpu = torch.randn(K, N, dtype=torch.float32)
ref_fp64 = (a_cpu.double() @ b_cpu.double()).float()

a_fp32 = a_cpu.to(device)
b_fp32 = b_cpu.to(device)
a_bf16 = a_fp32.to(torch.bfloat16)
b_bf16 = b_fp32.to(torch.bfloat16)

def eval_matmul(run_fn, label: str):
    for _ in range(5):
        out = run_fn()
    torch.accelerator.synchronize()

    iters = 25
    t0 = time.perf_counter()
    for _ in range(iters):
        out = run_fn()
    torch.accelerator.synchronize()
    dt = (time.perf_counter() - t0) / iters
    max_err = (out.float().cpu() - ref_fp64).abs().max().item()
    tflops = (2 * M * K * N / dt) / 1e12
    print(f"{label:<28} | {dt*1e3:6.3f} ms | {tflops:6.1f} TFLOPS | Max Abs Err: {max_err:.3e}")

eval_matmul(lambda: torch.matmul(a_bf16, b_bf16), "Native bfloat16 (1-pass)")

for mode_name, prec in [
    ("FP32 Precision.DEFAULT (1p)", torch.tpu.Precision.DEFAULT),
    ("FP32 Precision.HIGH    (3p)", torch.tpu.Precision.HIGH),
    ("FP32 Precision.HIGHEST (6p)", torch.tpu.Precision.HIGHEST),
]:
    def _runner(p=prec):
        with torch.tpu.precision(p):
            return torch.matmul(a_fp32, b_fp32)
    eval_matmul(_runner, mode_name)